# 10 · Bronze: ingest raw files

what it does: copy each CSV into a bronze table exactly as it is: every column, every row (including footer notes), every value as text.

Added lineage columns: `_row_num` (position in the file), `_source_file`, `_file_md5`, `_run_id`, `_ingested_at`.

In [0]:
%run ./00_common

In [0]:
def unique_names(cols):
    """Clean header names and de-duplicate them (the ABS income file repeats five headers)."""
    seen, out = {}, []
    for c in cols:
        n = clean_name(c)
        seen[n] = seen.get(n, 0) + 1
        out.append(n if seen[n] == 1 else f"{n}_{seen[n] - 1}")
    return out

def md5(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

manifest = []
for source, fname in FILES.items():
    path = f"{RAW_PATH}/{fname}"
    pdf = pd.read_csv(path, skiprows=HEADER_ROW[source], dtype=str, keep_default_na=False,
                      encoding="utf-8-sig", low_memory=False)
    original_headers = list(pdf.columns)
    pdf.columns = unique_names(pdf.columns)
    pdf.insert(0, "_row_num", range(1, len(pdf) + 1))
    file_hash = md5(path)
    df = (spark.createDataFrame(pdf)
          .withColumn("_source_file", F.lit(fname))
          .withColumn("_file_md5", F.lit(file_hash))
          .withColumn("_run_id", F.lit(RUN_ID))
          .withColumn("_ingested_at", F.current_timestamp()))
    n = write(df, "bronze", source)
    log_step("bronze", f"ingest_{source}", f"bronze.{source}", len(pdf), n, 0, f"{len(original_headers)} columns")
    manifest.append((RUN_ID, fname, os.path.getsize(path), file_hash, len(pdf), len(original_headers),
                     HEADER_ROW[source], datetime.now()))

### File manifest
One row per file per run: size, MD5 hash and row count. If a source file changes between runs, the hash changes, so you can always tell which version of the data a result came from.

In [0]:
mf = spark.createDataFrame(manifest, "run_id string, file string, bytes bigint, md5 string, rows bigint, columns int, header_row int, ingested_at timestamp")
mf.write.format(TABLE_FORMAT).mode("append").saveAsTable(tname("audit", "file_manifest"))
display(mf)

### Column name mapping (evidence for the source-to-target mapping table)

In [0]:
for source in FILES:
    cols = [c for c in read("bronze", source).columns if not c.startswith("_")]
    print(f"bronze.{source}: {len(cols)} columns, e.g. {cols[:6]}")